In [7]:
import pandas as pd
import numpy as np
from sklearn.metrics import silhouette_score, calinski_harabasz_score

In [8]:

file_path = 'simulationssimulation_results_summary_new.xlsx'
sheets = ['betweenness', 'degree', 'random']

In [9]:


# 手动定义这10列的名称，彻底绕开 Excel 双层表头合并单元格带来的解析混乱
col_names = [
    'City', 'Strategy', 'LCC_AUC', 'EFF_AUC', 
    'planA', 'planB_kmeans', 'planB_hierarchical', 
    'planC_kmeans', 'planC_gmm', 'planC_agg'
]

In [10]:
print("启动终极解析模式：无视表头结构，基于纯物理列号提取数据...\n")

for sheet in sheets:
    try:
        # header=None 让 Pandas 放弃思考表头，直接把 A列当第0列，B列当第1列，C列当第2列...
        df_raw = pd.read_excel(file_path, sheet_name=sheet, header=None)
        
        # 将 C列(2) 和 D列(3) 强转为数字。
        # 巧妙之处：前几行的表头文字（如 "LCC_AUC"）会自动变成 NaN，而真正的小数会被保留！
        lcc_col = pd.to_numeric(df_raw.iloc[:, 2], errors='coerce')
        eff_col = pd.to_numeric(df_raw.iloc[:, 3], errors='coerce')
        
        # 拼装成一个新的、完全受控的数据框
        clean_df = pd.DataFrame({
            'City': df_raw.iloc[:, 0],
            'LCC': lcc_col,
            'EFF': eff_col
        })
        
        # 提取您提到的分类结果列（为了防止漏掉，我将 E、F、G、H、I、J、K 列全部抓取）
        # 对应物理索引是 4 到 10
        col_letters = ['E', 'F', 'G', 'H', 'I', 'J', 'K']
        for i, col_idx in enumerate(range(4, 11)):
            if col_idx < df_raw.shape[1]: # 防止您的表格其实没有 K 列导致越界
                clean_df[f'列_{col_letters[i]}'] = df_raw.iloc[:, col_idx]

        # 【核心过滤】：删掉 C列或 D列是 NaN 的行！
        # 这一步非常丝滑，它会瞬间把最上面的几层表头、底部的空白行全部精准切除，只留下纯数据。
        clean_df = clean_df.dropna(subset=['LCC', 'EFF']).reset_index(drop=True)
        
        if clean_df.empty:
            print(f"❌ 警告: 工作簿 '{sheet}' 的 C 列和 D 列中找不到任何有效数字！请打开 Excel 确认数据是否真在 C 和 D 列。")
            continue

        # 准备数据供 sklearn 使用
        cities = clean_df['City'].astype(str).values
        X = clean_df[['LCC', 'EFF']].values
        
        evaluation_results = []
        cluster_details = {}

        # 遍历我们抓下来的 E 到 K 列
        plan_columns = [col for col in clean_df.columns if col.startswith('列_')]
        
        for col_name in plan_columns:
            raw_labels = clean_df[col_name]
            numeric_labels = pd.to_numeric(raw_labels, errors='coerce')
            
            # 自动修复类似 "太原填了 d" 的录入错误
            if numeric_labels.isna().any():
                valid_mask = numeric_labels.notna()
                if not valid_mask.any(): # 如果这一整列全是文字（比如不小心抓错列了），直接跳过
                    continue 
                current_X = X[valid_mask]
                current_labels = numeric_labels[valid_mask].astype(int).values
                current_cities = cities[valid_mask]
                print(f"⚠️ 提示: '{sheet}' 的 [{col_name}] 中发现非数字（已自动剔除该节点）。")
            else:
                current_X = X
                current_labels = numeric_labels.astype(int).values
                current_cities = cities

            unique_labels = np.unique(current_labels)
            
            # 只有当分类数大于1，且小于城市总数时，才计算指标
            if 1 < len(unique_labels) < len(current_labels):
                sil_score = silhouette_score(current_X, current_labels, metric='euclidean')
                ch_score = calinski_harabasz_score(current_X, current_labels)
                
                groups = {}
                for cluster_id in unique_labels:
                    members = current_cities[current_labels == cluster_id]
                    groups[cluster_id] = list(members)
                cluster_details[col_name] = groups
            else:
                sil_score = np.nan
                ch_score = np.nan
                
            # 如果分数有效，才计入结果表
            if pd.notna(sil_score):
                evaluation_results.append({
                    '分类所在列': col_name,
                    '轮廓系数': round(sil_score, 4),
                    'CH 指数': round(ch_score, 4)
                })
                
        # --- 打印输出排版 ---
        if not evaluation_results:
            print(f"工作簿 '{sheet}' 中没有找到有效的分类数据列。\n")
            continue
            
        print(f"\n========== 【 {sheet.upper()} 攻击策略 】 ==========")
        results_df = pd.DataFrame(evaluation_results)
        print(results_df.to_string(index=False))
        print("-" * 50)
        
        # 选出最高分
        best_plan_idx = results_df['CH 指数'].astype(float).idxmax()
        best_plan_name = results_df.loc[best_plan_idx, '分类所在列']
        
        print(f"🏆 推荐采纳: [ Excel {best_plan_name} ] (因其 CH 指数最高，类间区分度最好)")
        print(f"📊 该方案的城市分组名单如下：")
        for cluster_id, members in cluster_details[best_plan_name].items():
            print(f"  ▶ Cluster {cluster_id} (共 {len(members)} 城): {', '.join(members)}")
        print("\n" + "="*60 + "\n")

    except Exception as e:
        print(f"处理工作簿 '{sheet}' 时发生未知异常: {e}\n")

启动终极解析模式：无视表头结构，基于纯物理列号提取数据...


========== 【 BETWEENNESS 攻击策略 】 ==========
分类所在列    轮廓系数  CH 指数
  列_F -0.2126 4.6251
  列_G -0.0692 0.2361
  列_H -0.0019 7.0643
  列_I -0.3017 0.6908
  列_J -0.0937 0.0870
  列_K -0.1013 1.7474
--------------------------------------------------
🏆 推荐采纳: [ Excel 列_H ] (因其 CH 指数最高，类间区分度最好)
📊 该方案的城市分组名单如下：
  ▶ Cluster 0 (共 13 城): Beijing, Changchun, Guiyang, Hohhot, Kunming, Lanzhou, Luoyang, Nanning, Shenzhen, Taiyuan, Tianjin, urumchi, Xiamen
  ▶ Cluster 1 (共 29 城): Changsha, Changzhou, Chengdu, Dalian, Dongguan, Foshan, Fuzhou, Guangzhou, Hangzhou, Harbin, Hefei, Jinan, Nanchang, nanking, Nantong, ningbo, Qingdao, Shanghai, Shaoxing, Shenyang, Shijiazhuang, sian, Suzhou, Wenzhou, Wuhan, Wuhu, Wuxi, Xuzhou, Zhengzhou
  ▶ Cluster 2 (共 2 城): Chongqing, Chuzhou



========== 【 DEGREE 攻击策略 】 ==========
分类所在列    轮廓系数   CH 指数
  列_F -0.1743 13.3118
  列_G -0.0672  1.0793
  列_H  0.1338  8.1338
  列_I -0.2602  0.3486
  列_J -0.0854  0.9696
  列_K -0.0736  0.6546
---------